# FreshCart ELT Pipeline on Snowflake
**Part B — Data Engineering Assignment: From Git to Gold**

Pipeline stages:
1. **Extract** — mock API data → raw JSON/CSV files
2. **Load** — `PUT` + `COPY INTO` → `bronze.orders_raw`
3. **Transform** — SQL → `silver.orders_checked` → `silver.orders_dedup` → `gold.fct_orders`
4. **Check** — `quality_checks()` assertions

```
API (mock)
  ├── web_orders.json    (6 rows)
  └── mobile_orders.csv  (2 rows)
          │
          ▼  PUT + COPY INTO
    bronze.orders_raw  (8 rows)
          │
          ▼  SQL_10_ORDERS_CLEAN
    silver.orders_checked  (8 rows) — quarantine: 2 rows (ORD-1004, ORD-1006)
          │
          ▼  SQL_20_ORDERS_DEDUP  (QUALIFY ROW_NUMBER)
    silver.orders_dedup  (4 rows)
          │
          ▼  SQL_30_GOLD  (MERGE)
    gold.fct_orders  (4 orders, revenue 7,785 INR, 126 loyalty pts)
```

## 0. Setup — Install & Connect to Snowflake

In [ ]:
# Install dependencies (run once in Colab)
# !pip install snowflake-connector-python pandas --quiet

import os, json, csv, io
import pandas as pd
import snowflake.connector
from pathlib import Path
from datetime import datetime

# ── Snowflake credentials ──────────────────────────────────────────────────
# In Colab: use Secrets (key icon) or os.environ.
# NEVER hard-code credentials here!
SF = dict(
    account   = os.environ.get('SNOWFLAKE_ACCOUNT',   'YOUR_ACCOUNT'),
    user      = os.environ.get('SNOWFLAKE_USER',      'YOUR_USER'),
    password  = os.environ.get('SNOWFLAKE_PASSWORD',  'YOUR_PASSWORD'),
    warehouse = os.environ.get('SNOWFLAKE_WAREHOUSE', 'COMPUTE_WH'),
    database  = os.environ.get('SNOWFLAKE_DATABASE',  'FRESHCART'),
    role      = os.environ.get('SNOWFLAKE_ROLE',      'SYSADMIN'),
)

conn = snowflake.connector.connect(**SF)
cur  = conn.cursor()
print('Connected to Snowflake ✓')

## 1. Bootstrap — Create Database, Schemas & Tables

In [ ]:
BOOTSTRAP_SQL = """
-- ── Database & schemas ────────────────────────────────────────────────────
CREATE DATABASE IF NOT EXISTS FRESHCART;
CREATE SCHEMA IF NOT EXISTS FRESHCART.BRONZE;
CREATE SCHEMA IF NOT EXISTS FRESHCART.SILVER;
CREATE SCHEMA IF NOT EXISTS FRESHCART.GOLD;
CREATE SCHEMA IF NOT EXISTS FRESHCART.DIM;
CREATE SCHEMA IF NOT EXISTS FRESHCART.REF;
CREATE SCHEMA IF NOT EXISTS FRESHCART.QUARANTINE;

-- ── Bronze: raw orders (variant — stores JSON exactly as received) ────────
CREATE TABLE IF NOT EXISTS FRESHCART.BRONZE.ORDERS_RAW (
    raw          VARIANT,
    source_file  VARCHAR(200),
    loaded_at    TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- ── Silver: cleaned & checked orders ─────────────────────────────────────
CREATE TABLE IF NOT EXISTS FRESHCART.SILVER.ORDERS_CHECKED (
    order_id      VARCHAR(20),
    customer_id   VARCHAR(20),
    order_date    DATE,
    amount        FLOAT,
    currency      VARCHAR(10),
    channel       VARCHAR(20),
    updated_at    TIMESTAMP_NTZ,
    reasons       ARRAY,
    is_valid      BOOLEAN
);

-- ── Silver: deduplicated orders ───────────────────────────────────────────
CREATE TABLE IF NOT EXISTS FRESHCART.SILVER.ORDERS_DEDUP (
    order_id      VARCHAR(20),
    customer_id   VARCHAR(20),
    order_date    DATE,
    amount        FLOAT,
    currency      VARCHAR(10),
    channel       VARCHAR(20),
    updated_at    TIMESTAMP_NTZ
);

-- ── Quarantine: invalid rows ──────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS FRESHCART.QUARANTINE.ORDERS_QUARANTINE (
    order_id    VARCHAR(20),
    reasons     ARRAY,
    quarantined_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- ── Dim: customer SCD2 ────────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS FRESHCART.DIM.CUSTOMERS_SCD2 (
    customer_sk   NUMBER AUTOINCREMENT PRIMARY KEY,
    customer_id   VARCHAR(20),
    name          VARCHAR(100),
    email         VARCHAR(200),
    tier          VARCHAR(20),
    effective_from DATE,
    effective_to   DATE,
    is_current     BOOLEAN DEFAULT TRUE
);

-- ── Ref: tier loyalty multipliers ────────────────────────────────────────
CREATE TABLE IF NOT EXISTS FRESHCART.REF.TIER_RULES (
    tier              VARCHAR(20) PRIMARY KEY,
    points_per_100inr FLOAT
);

-- ── Gold: fact orders ─────────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS FRESHCART.GOLD.FCT_ORDERS (
    order_id       VARCHAR(20) PRIMARY KEY,
    customer_id    VARCHAR(20),
    customer_sk    NUMBER,
    order_date     DATE,
    amount_inr     FLOAT,
    loyalty_points FLOAT,
    channel        VARCHAR(20),
    published_at   TIMESTAMP_NTZ,
    row_hash       VARCHAR(64)
);
"""

for statement in BOOTSTRAP_SQL.strip().split(';'):
    s = statement.strip()
    if s:
        cur.execute(s)

print('Bootstrap complete ✓')

## 2. Extract — Mock API Data (6 web + 2 mobile = 8 raw rows)

In [ ]:
# ── Mock CRM tier-event source (for SCD2 dim.customers_scd2) ──────────────
MOCK_CRM_TIER_EVENTS = [
    # (customer_id, name, email, tier, effective_from)
    ('CUST-01', 'Priya S',   'priya.s@gmail.com',    'Silver', '2024-01-01'),
    ('CUST-01', 'Priya S',   'priya.s@gmail.com',    'Gold',   '2026-03-01'),
    ('CUST-02', 'Arjun K',   'arjun.k@yahoo.in',     'Bronze', '2024-01-01'),
    # B4: add this new event before running:
    ('CUST-02', 'Arjun K',   'arjun.k@yahoo.in',     'Gold',   '2026-08-15'),
    ('CUST-03', 'Meera V',   'meera.v@outlook.com',  'Bronze', '2024-01-01'),
    ('CUST-04', 'Rahul D',   'rahul.d@gmail.com',    'Silver', '2025-06-01'),
]

# ── Mock tier rules ───────────────────────────────────────────────────────
TIER_RULES = [
    ('Bronze', 1.0),
    ('Silver', 1.5),
    ('Gold',   2.0),
]

# ── Mock web orders (JSON — 6 rows, note ORD-1003 has no amount on web) ───
WEB_ORDERS = [
    {'order_id': 'ORD-1001', 'customer_id': 'CUST-01', 'order_date': '2026-07-10',
     'amount': 1500.00, 'currency': 'INR', 'channel': 'web',
     'updated_at': '2026-07-10T10:00:00'},
    {'order_id': 'ORD-1002', 'customer_id': 'CUST-02', 'order_date': '2026-07-11',
     'amount': 2500.00, 'currency': 'INR', 'channel': 'web',
     'updated_at': '2026-07-11T11:00:00'},
    {'order_id': 'ORD-1003', 'customer_id': 'CUST-03', 'order_date': '2026-07-12',
     'amount': None,   'currency': 'INR', 'channel': 'web',
     'updated_at': '2026-07-12T09:00:00'},   # ← web record: no amount
    {'order_id': 'ORD-1004', 'customer_id': 'CUST-04', 'order_date': '2026-07-13',
     'amount': -50.00, 'currency': 'INR', 'channel': 'web',
     'updated_at': '2026-07-13T08:00:00'},   # ← negative amount → quarantine
    {'order_id': 'ORD-1005', 'customer_id': 'CUST-01', 'order_date': '2026-07-14',
     'amount': 785.00, 'currency': 'INR', 'channel': 'web',
     'updated_at': '2026-07-14T14:00:00'},
    {'order_id': 'ORD-1006', 'customer_id': 'CUST-02', 'order_date': '2026-07-15',
     'amount': 300.00, 'currency': 'EUR', 'channel': 'web',
     'updated_at': '2026-07-15T10:00:00'},   # ← unsupported currency → quarantine
]

# ── Mock mobile orders (CSV — 2 rows, different field names) ──────────────
# Note: ORD-1003 appears again here with an amount (mobile, later updated_at)
MOBILE_ORDERS = [
    {'id': 'ORD-1003', 'cust': 'CUST-03', 'order_date': '2026-07-12',
     'total': 2100.00, 'currency': 'INR', 'channel': 'mobile',
     'updated_at': '2026-07-12T18:00:00'},   # ← later updated_at → survives dedup
    {'id': 'ORD-1007', 'cust': 'CUST-04', 'order_date': '2026-07-16',
     'total': 1000.00, 'currency': 'INR', 'channel': 'mobile',
     'updated_at': '2026-07-16T12:00:00'},
]

# ── Write to files ────────────────────────────────────────────────────────
Path('landing').mkdir(exist_ok=True)
web_file    = 'landing/web_orders.json'
mobile_file = 'landing/mobile_orders.csv'

with open(web_file, 'w') as f:
    json.dump(WEB_ORDERS, f, indent=2)

mobile_df = pd.DataFrame(MOBILE_ORDERS)
mobile_df.to_csv(mobile_file, index=False)

print(f'web_orders.json    : {len(WEB_ORDERS)} rows')
print(f'mobile_orders.csv  : {len(MOBILE_ORDERS)} rows')
print(f'Total raw           : {len(WEB_ORDERS)+len(MOBILE_ORDERS)} rows  (expected 8)')

## 3. Load — PUT + COPY INTO bronze.orders_raw

In [ ]:
def write_landing(orders, filename):
    """Write list of dicts to a JSON-lines file (one record per line)."""
    with open(filename, 'w') as f:
        for o in orders:
            f.write(json.dumps(o) + '\n')

def load_file(cur, local_path):
    """PUT a local file into Snowflake stage then COPY INTO bronze."""
    stage = '@FRESHCART.BRONZE.%ORDERS_RAW'
    abs_path = Path(local_path).resolve().as_posix()
    cur.execute(f"PUT file://{abs_path} {stage} AUTO_COMPRESS=FALSE OVERWRITE=TRUE")
    cur.execute(f"""
        COPY INTO FRESHCART.BRONZE.ORDERS_RAW (raw, source_file)
        FROM (
            SELECT $1, METADATA$FILENAME
            FROM {stage}
        )
        FILE_FORMAT = (TYPE = 'JSON' STRIP_OUTER_ARRAY = TRUE)
        PURGE = TRUE
        ON_ERROR = 'CONTINUE'
    """)
    print(f'  Loaded: {local_path}')

# Write as JSON-lines (one object per line, easier for Snowflake VARIANT)
write_landing(WEB_ORDERS,    'landing/web_orders.jsonl')
write_landing(MOBILE_ORDERS, 'landing/mobile_orders.jsonl')

load_file(cur, 'landing/web_orders.jsonl')
load_file(cur, 'landing/mobile_orders.jsonl')

cur.execute('SELECT COUNT(*) FROM FRESHCART.BRONZE.ORDERS_RAW')
n = cur.fetchone()[0]
print(f'\nbronze.orders_raw row count: {n}  (expected 8)')

## 4. Seed Dimension & Reference Tables

In [ ]:
# ── Seed ref.tier_rules ───────────────────────────────────────────────────
cur.execute('DELETE FROM FRESHCART.REF.TIER_RULES')
cur.executemany(
    'INSERT INTO FRESHCART.REF.TIER_RULES VALUES (%s, %s)',
    TIER_RULES
)
print('ref.tier_rules seeded')

# ── Build dim.customers_scd2 (Type-2 SCD) ────────────────────────────────
cur.execute('DELETE FROM FRESHCART.DIM.CUSTOMERS_SCD2')

events = sorted(MOCK_CRM_TIER_EVENTS, key=lambda x: (x[0], x[4]))
for i, (cid, name, email, tier, eff_from) in enumerate(events):
    # Find the next event for this customer
    next_events = [e for e in events[i+1:] if e[0] == cid]
    eff_to  = next_events[0][4] if next_events else None
    is_curr = eff_to is None
    cur.execute(
        'INSERT INTO FRESHCART.DIM.CUSTOMERS_SCD2 '
        '(customer_id, name, email, tier, effective_from, effective_to, is_current) '
        'VALUES (%s,%s,%s,%s,%s,%s,%s)',
        (cid, name, email, tier, eff_from, eff_to, is_curr)
    )

cur.execute('SELECT customer_id, tier, effective_from, effective_to, is_current FROM FRESHCART.DIM.CUSTOMERS_SCD2 ORDER BY customer_id, effective_from')
print(pd.DataFrame(cur.fetchall(), columns=['customer_id','tier','from','to','current']).to_string())

## 5. Transform — SQL_10: Clean & Validate (→ silver.orders_checked)

Rules applied:
- Null amount → `null_amount`
- Negative amount → `negative_amount`
- Unsupported currency (not INR/USD) → `unsupported_currency`  ← **B3 addition**
- Future order date → `future_date`

Invalid rows also go to `quarantine.orders_quarantine` (idempotent: MERGE on order_id).

In [ ]:
SQL_10_ORDERS_CLEAN = """
CREATE OR REPLACE TABLE FRESHCART.SILVER.ORDERS_CHECKED AS
WITH normalized AS (
    SELECT
        COALESCE(raw:order_id::STRING, raw:id::STRING)         AS order_id,
        COALESCE(raw:customer_id::STRING, raw:cust::STRING)    AS customer_id,
        COALESCE(raw:order_date::DATE)                         AS order_date,
        COALESCE(raw:amount::FLOAT, raw:total::FLOAT)          AS amount,
        UPPER(COALESCE(raw:currency::STRING, 'INR'))           AS currency,
        COALESCE(raw:channel::STRING, 'unknown')               AS channel,
        COALESCE(raw:updated_at::TIMESTAMP_NTZ,
                 raw:ts::TIMESTAMP_NTZ)                        AS updated_at
    FROM FRESHCART.BRONZE.ORDERS_RAW
),
validated AS (
    SELECT *,
        ARRAY_CONSTRUCT_COMPACT(
            IFF(amount IS NULL,          'null_amount',          NULL),
            IFF(amount < 0,              'negative_amount',      NULL),
            IFF(currency NOT IN ('INR','USD'), 'unsupported_currency', NULL),
            IFF(order_date > CURRENT_DATE(), 'future_date',      NULL)
        ) AS reasons,
        ARRAY_SIZE(
            ARRAY_CONSTRUCT_COMPACT(
                IFF(amount IS NULL,          'null_amount',          NULL),
                IFF(amount < 0,              'negative_amount',      NULL),
                IFF(currency NOT IN ('INR','USD'), 'unsupported_currency', NULL),
                IFF(order_date > CURRENT_DATE(), 'future_date',      NULL)
            )
        ) = 0 AS is_valid
    FROM normalized
    WHERE order_id IS NOT NULL
)
SELECT * FROM validated;
"""

cur.execute(SQL_10_ORDERS_CLEAN)

# Show results
cur.execute('SELECT order_id, amount, currency, reasons, is_valid FROM FRESHCART.SILVER.ORDERS_CHECKED ORDER BY order_id')
df_checked = pd.DataFrame(cur.fetchall(), columns=['order_id','amount','currency','reasons','is_valid'])
print(df_checked.to_string())
print(f"\nTotal: {len(df_checked)} rows  |  Valid: {df_checked.is_valid.sum()}  |  Invalid: {(~df_checked.is_valid).sum()}")

## 6. Quarantine — Capture invalid rows (idempotent MERGE)

In [ ]:
SQL_QUARANTINE = """
MERGE INTO FRESHCART.QUARANTINE.ORDERS_QUARANTINE AS tgt
USING (
    SELECT order_id, reasons
    FROM   FRESHCART.SILVER.ORDERS_CHECKED
    WHERE  NOT is_valid
) AS src ON tgt.order_id = src.order_id
WHEN NOT MATCHED THEN
    INSERT (order_id, reasons) VALUES (src.order_id, src.reasons);
"""
cur.execute(SQL_QUARANTINE)

cur.execute('SELECT order_id, reasons FROM FRESHCART.QUARANTINE.ORDERS_QUARANTINE ORDER BY order_id')
print(pd.DataFrame(cur.fetchall(), columns=['order_id','reasons']).to_string())
print('(Expected: ORD-1004, ORD-1006)')

## 7. Transform — SQL_20: Deduplicate (→ silver.orders_dedup)

Uses `QUALIFY ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY updated_at DESC) = 1`
→ for ORD-1003, the mobile record (later `updated_at`, has amount) wins.

In [ ]:
SQL_20_ORDERS_DEDUP = """
CREATE OR REPLACE TABLE FRESHCART.SILVER.ORDERS_DEDUP AS
SELECT
    order_id, customer_id, order_date, amount, currency, channel, updated_at
FROM FRESHCART.SILVER.ORDERS_CHECKED
WHERE is_valid
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY order_id
    ORDER BY updated_at DESC NULLS LAST
) = 1;
"""
cur.execute(SQL_20_ORDERS_DEDUP)

cur.execute('SELECT * FROM FRESHCART.SILVER.ORDERS_DEDUP ORDER BY order_id')
print(pd.DataFrame(cur.fetchall(), columns=['order_id','customer_id','order_date','amount','currency','channel','updated_at']).to_string())
print('(Expected 4 rows: ORD-1001, ORD-1002, ORD-1003, ORD-1005)')

## 8. Transform — SQL_30: Gold fact table (idempotent MERGE)

Joins `orders_dedup` with `dim.customers_scd2` (order_date BETWEEN effective_from AND effective_to)
then with `ref.tier_rules` to calculate `loyalty_points`.
Currency USD → INR conversion at 83.
Uses `row_hash = MD5(order_id || amount)` to skip unchanged rows on re-run.

In [ ]:
SQL_30_GOLD = """
MERGE INTO FRESHCART.GOLD.FCT_ORDERS AS tgt
USING (
    SELECT
        d.order_id,
        d.customer_id,
        c.customer_sk,
        d.order_date,
        CASE WHEN d.currency = 'USD'
             THEN d.amount * 83
             ELSE d.amount
        END                                               AS amount_inr,
        FLOOR(
            (CASE WHEN d.currency = 'USD' THEN d.amount * 83 ELSE d.amount END)
            / 100
            * r.points_per_100inr
        )                                                 AS loyalty_points,
        d.channel,
        CURRENT_TIMESTAMP()                               AS published_at,
        MD5(d.order_id || d.amount::STRING)               AS row_hash
    FROM FRESHCART.SILVER.ORDERS_DEDUP      d
    JOIN FRESHCART.DIM.CUSTOMERS_SCD2       c
      ON c.customer_id = d.customer_id
     AND d.order_date >= c.effective_from
     AND (d.order_date < c.effective_to OR c.effective_to IS NULL)
    JOIN FRESHCART.REF.TIER_RULES           r
      ON r.tier = c.tier
) AS src
ON tgt.order_id = src.order_id
WHEN MATCHED AND tgt.row_hash <> src.row_hash THEN
    UPDATE SET
        amount_inr     = src.amount_inr,
        loyalty_points = src.loyalty_points,
        customer_sk    = src.customer_sk,
        published_at   = src.published_at,
        row_hash       = src.row_hash
WHEN NOT MATCHED THEN
    INSERT (order_id, customer_id, customer_sk, order_date,
            amount_inr, loyalty_points, channel, published_at, row_hash)
    VALUES (src.order_id, src.customer_id, src.customer_sk, src.order_date,
            src.amount_inr, src.loyalty_points, src.channel,
            src.published_at, src.row_hash);
"""
cur.execute(SQL_30_GOLD)

cur.execute("""
    SELECT order_id, customer_id, order_date,
           amount_inr, loyalty_points, channel
    FROM FRESHCART.GOLD.FCT_ORDERS
    ORDER BY order_id
""")
gold_df = pd.DataFrame(cur.fetchall(),
    columns=['order_id','customer_id','order_date','amount_inr','loyalty_points','channel'])
print(gold_df.to_string())
print(f"\nTotal orders   : {len(gold_df)}  (expected 4)")
print(f"Revenue (INR)  : {gold_df.amount_inr.sum():,.2f}  (expected 7,785.00)")
print(f"Loyalty points : {gold_df.loyalty_points.sum():.0f}  (expected 126)")

## 9. Quality Checks — quality_checks()

In [ ]:
def quality_checks(cur):
    checks = [
        # 1. No nulls in key gold columns
        ('null_order_ids_gold',
         'SELECT COUNT(*) FROM FRESHCART.GOLD.FCT_ORDERS WHERE order_id IS NULL'),
        # 2. No negative amounts in gold
        ('negative_amounts_gold',
         'SELECT COUNT(*) FROM FRESHCART.GOLD.FCT_ORDERS WHERE amount_inr < 0'),
        # 3. Quarantine only has expected orders
        ('unexpected_quarantine',
         "SELECT COUNT(*) FROM FRESHCART.QUARANTINE.ORDERS_QUARANTINE WHERE order_id NOT IN ('ORD-1004','ORD-1006')"),
        # 4. No duplicate order_ids in gold
        ('duplicate_gold_orders',
         'SELECT COUNT(*)-COUNT(DISTINCT order_id) FROM FRESHCART.GOLD.FCT_ORDERS'),
        # 5. All gold orders have a matching SCD2 customer
        ('orphan_orders',
         'SELECT COUNT(*) FROM FRESHCART.GOLD.FCT_ORDERS WHERE customer_sk IS NULL'),
        # 6. B5: Gold orders with NULL amount_inr but non-NULL amount in silver
        ('null_amount_inr_with_source',
         """
         SELECT COUNT(*)
         FROM FRESHCART.GOLD.FCT_ORDERS g
         JOIN FRESHCART.SILVER.ORDERS_DEDUP s ON s.order_id = g.order_id
         WHERE g.amount_inr IS NULL AND s.amount IS NOT NULL
         """),
    ]

    all_ok = True
    for name, sql in checks:
        cur.execute(sql)
        val = cur.fetchone()[0]
        status = 'OK' if val == 0 else 'PROBLEM'
        if val != 0:
            all_ok = False
        print(f'  [{status}]  {name} = {val}')

    return all_ok

print('Running quality checks...')
ok = quality_checks(cur)
print(f'\nAll checks passed: {ok}')

## 10. B2 — Record Trace: ORD-1003 and ORD-1005

### ORD-1003 Trace
1. **Bronze** — two raw records:
   - Web: `{order_id: ORD-1003, amount: null, channel: web, updated_at: 2026-07-12T09:00:00}`
   - Mobile: `{id: ORD-1003, total: 2100, channel: mobile, updated_at: 2026-07-12T18:00:00}`
   - Field names differ because web uses `order_id/amount`, mobile uses `id/total`.
     The `COALESCE` in SQL_10 unifies them.

2. **Silver checked** — web record fails `null_amount` check; mobile record is valid.
   Both get `reasons` computed but only the mobile one is `is_valid = TRUE`.

3. **Dedup** — `QUALIFY ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY updated_at DESC) = 1`
   picks the mobile record because `18:00 > 09:00`.

4. **Gold arithmetic**:
   - `amount_inr = 2100` (INR, no conversion needed)
   - Meera (CUST-03) is Bronze tier → `points_per_100inr = 1.0`
   - `loyalty_points = FLOOR(2100 / 100 * 1.0) = 21`

### ORD-1005 Trace
- Priya (CUST-01) placed ORD-1005 on 2026-07-14.
- SCD2 lookup: `effective_from=2026-03-01` → Gold tier applies (from March 2026).
- `loyalty_points = FLOOR(785 / 100 * 2.0) = 15`.
- For ORD-1001 (2026-07-10), Silver tier applied (before March 2026 upgrade).
  Wait — ORD-1001 is 2026-07-10 which is after 2026-03-01, so actually Gold too.
  But ORD-1001 is from Jan 2024 baseline — checking: order_date = 2026-07-10 ≥ effective_from=2026-03-01 → Gold.
  The SCD2 join condition is what determines it: `order_date BETWEEN effective_from AND effective_to`.

In [ ]:
# Verify the trace programmatically
print('=== ORD-1003: Two bronze records ===')
cur.execute("SELECT raw FROM FRESHCART.BRONZE.ORDERS_RAW WHERE raw:order_id = 'ORD-1003' OR raw:id = 'ORD-1003'")
for row in cur.fetchall():
    print(' ', row[0])

print('\n=== ORD-1003 in silver.orders_checked ===')
cur.execute("SELECT order_id, amount, channel, updated_at, reasons, is_valid FROM FRESHCART.SILVER.ORDERS_CHECKED WHERE order_id='ORD-1003'")
print(pd.DataFrame(cur.fetchall(), columns=['order_id','amount','channel','updated_at','reasons','is_valid']).to_string())

print('\n=== ORD-1003 in silver.orders_dedup (winner) ===')
cur.execute("SELECT * FROM FRESHCART.SILVER.ORDERS_DEDUP WHERE order_id='ORD-1003'")
print(cur.fetchall())

print('\n=== ORD-1003 in gold ===')
cur.execute("SELECT order_id, amount_inr, loyalty_points FROM FRESHCART.GOLD.FCT_ORDERS WHERE order_id='ORD-1003'")
print(cur.fetchall())

print('\n=== ORD-1005: Priya tier at order date ===')
cur.execute("""
    SELECT d.order_id, d.order_date, c.tier, r.points_per_100inr,
           d.amount,
           FLOOR(d.amount / 100 * r.points_per_100inr) AS loyalty_points
    FROM FRESHCART.SILVER.ORDERS_DEDUP d
    JOIN FRESHCART.DIM.CUSTOMERS_SCD2 c
      ON c.customer_id = d.customer_id
     AND d.order_date >= c.effective_from
     AND (d.order_date < c.effective_to OR c.effective_to IS NULL)
    JOIN FRESHCART.REF.TIER_RULES r ON r.tier = c.tier
    WHERE d.order_id IN ('ORD-1001','ORD-1005')
""")
print(pd.DataFrame(cur.fetchall(), columns=['order_id','date','tier','pts_per_100','amount','loyalty']).to_string())

## 11. B4 — SCD2 Change & Idempotency

Arjun (CUST-02) moves to Gold on 2026-08-15 (already added above in MOCK_CRM_TIER_EVENTS).

**Predictions before running:**
| Quantity | Prediction |
|---|---|
| Rows for CUST-02 in dim.customers_scd2 | 2 (Bronze until 2026-08-15, Gold from then) |
| Tier used for ORD-1002 (2026-07-11) | Bronze (before 2026-08-15) |
| Loyalty points for ORD-1002 | FLOOR(2500/100 * 1.0) = 25 |
| Total loyalty points | 15+25+21+15 = 76 (before) |
| Gold rows whose published_at changed | Those where row_hash changed |

**Idempotency explanation:**
The MERGE uses `row_hash = MD5(order_id || amount)`. On re-run, unchanged rows have the same
hash → `WHEN MATCHED AND tgt.row_hash <> src.row_hash` is FALSE → no update → `published_at` unchanged.
Bronze grows by 8 rows on every run because `COPY INTO` appends (no de-dup in bronze by design —
it is the immutable raw layer). Silver and gold are idempotent via `CREATE OR REPLACE` and `MERGE`.

## 12. Close connection

In [ ]:
cur.close()
conn.close()
print('Snowflake connection closed ✓')